In [ ]:
import sys
from pathlib import Path

def find_gptcast_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'gptcast').is_dir():
            return p
        if (p / 'GPTCast' / 'gptcast').is_dir():
            return (p / 'GPTCast').resolve()
        p = p.parent
    return start.resolve()

ROOT = find_gptcast_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Using ROOT:', ROOT)

from gptcast.models import VAEGANVQ
from gptcast.data import Era5LandSwvl1DataModule
from gptcast.utils.plotting import plot_mutiple_era5land as plot_mutiple
from gptcast.utils.converters import swvl1_norm_to_phys
import numpy as np
import random
from datetime import datetime
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from einops import rearrange

# Reproducibility: crops / random ops can introduce nondeterminism.
try:
    from lightning.pytorch import seed_everything
    seed_everything(42, workers=True)
except Exception:
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


## Load pretrained model and a soil moisture (swvl1) sequence of data from the dataset


In [ ]:
# ROOT is set in the first cell (auto-detected)

# Compare two first-stage checkpoints (update paths as needed)
#
# Upstream GPTCast notebook semantics: `old=vae_mae` and `new=vae_mwae` (downloaded from Zenodo).
# In this SWVL1 fork we do not download upstream precipitation models.
# Here, `old/new` refer to an earlier vs later checkpoint from the SAME SWVL1 MWAE first-stage run.
ae_old = VAEGANVQ.load_from_pretrained(str(ROOT / 'logs/train/runs/2026-03-01_18-29-57/checkpoints/epoch_038.ckpt'), device=device).to(device).eval()
ae_new = VAEGANVQ.load_from_pretrained(str(ROOT / 'logs/train/runs/2026-03-01_18-29-57/checkpoints/last.ckpt'), device=device).to(device).eval()


In [ ]:
# ROOT is set in the first cell (auto-detected)

md = Era5LandSwvl1DataModule(
    base_dir=str(ROOT / 'data/0.1/1/land_surface'),
    train_metadata_path_or_df=str(ROOT / 'data/0.1/era5land_swvl1_train.csv'),
    val_metadata_path_or_df=str(ROOT / 'data/0.1/era5land_swvl1_val.csv'),
    test_metadata_path_or_df=str(ROOT / 'data/0.1/era5land_swvl1_test.csv'),
    clip_and_normalize=(0.0, 0.8, -1.0, 1.0),
    resize=720,
    crop=256,
    smart_crop=False,
    max_mask_fraction=0.40,
    smart_crop_attempts=50,
    center_crop_val=False,
    random_rotate90=False,
    seq_len=7,
    stack_seq=None,
    batch_size=1,
    num_workers=0,
    pin_memory=False,
)
md.setup(stage='test')
tdl = md.test_dataloader()
itr = iter(tdl)

# Pick a land-heavy crop for visualization (avoid mostly-ocean patches).
MAX_TRIES = 200
MAX_MASK_FRAC = 0.40

best_el = None
best_frac = 1.0
selected_frac = None
for _ in range(MAX_TRIES):
    el = next(itr)
    frac = float(el['mask'].cpu().numpy().mean())
    if frac < best_frac:
        best_frac = frac
        best_el = el
    if frac <= MAX_MASK_FRAC:
        selected_frac = frac
        break
if selected_frac is None:
    el = best_el
    selected_frac = best_frac
    print(f"Warning: could not find a crop with mask_frac<={MAX_MASK_FRAC} in {MAX_TRIES} tries; using best found {selected_frac:.3f}")

print(f"Selected sample mask_frac={selected_frac:.3f}  file={el['file_path_'][0]}")


In [ ]:
dt = datetime.strptime(el['file_path_'][0], 'SM_%Y%m%d%H%M')
mask = el['mask'].cpu().numpy().squeeze()
data = rearrange(el['image'].cpu().numpy().squeeze(), 'h w s -> s h w')
input_data = np.ma.array(data, mask=np.broadcast_to(mask, data.shape))


## Reconstruct the input sequence with both autencoders and compare the results


In [ ]:
def reconstruct_swvl1(ae: VAEGANVQ, arr: np.ma.MaskedArray):
    assert arr.ndim == 3
    x = arr.data.astype(np.float32, copy=False)
    # Treat time as batch dimension (S,1,H,W)
    x = torch.tensor(x, dtype=torch.float32, device=device)[:, None, ...]
    with torch.no_grad():
        y, _ = ae(x, auto_pad=True)
    y = y.detach().cpu().numpy().squeeze().clip(-1, 1)
    return np.ma.array(y, mask=arr.mask)

new_rec = reconstruct_swvl1(ae_new, input_data)
old_rec = reconstruct_swvl1(ae_old, input_data)


### plot input, new and old autoencoder output


In [ ]:
plot_mutiple(swvl1_norm_to_phys(input_data), title=f"Input {dt}", figsize=(10,3))
plot_mutiple(swvl1_norm_to_phys(new_rec), title=f"Reconstructed (ckpt new)", figsize=(10,3))
plot_mutiple(swvl1_norm_to_phys(old_rec), title=f"Reconstructed (ckpt old)", figsize=(10,3))

# Quantitative check (masked MAE/RMSE in physical units m3/m3)
def per_lead_mae_rmse(pred: np.ma.MaskedArray, tgt: np.ma.MaskedArray):
    diff = pred - tgt
    mae = np.ma.mean(np.abs(diff), axis=(1, 2))
    rmse = np.sqrt(np.ma.mean(diff ** 2, axis=(1, 2)))
    return mae, rmse

inp_phys = swvl1_norm_to_phys(input_data)
new_phys = swvl1_norm_to_phys(new_rec)
old_phys = swvl1_norm_to_phys(old_rec)

mae_n, rmse_n = per_lead_mae_rmse(new_phys, inp_phys)
mae_o, rmse_o = per_lead_mae_rmse(old_phys, inp_phys)

print(f"Reconstruction metrics over {inp_phys.shape[0]} frames (masked/ocean ignored):")
print(f"  new_ckpt: mean MAE={float(mae_n.mean()):.4f}, mean RMSE={float(rmse_n.mean()):.4f}")
print(f"  old_ckpt: mean MAE={float(mae_o.mean()):.4f}, mean RMSE={float(rmse_o.mean()):.4f}")
